# Machine learning: problems, evidence, and real data

**Lecture 9 · Notebook 00 · CMOR 438 / INDE 577**  
**Core:** 95 minutes · **Practice:** 40 minutes · **Extension:** 30+ minutes

Machine learning is not one algorithm. It is a family of ways to learn predictive, descriptive, generative, or decision behavior from data and feedback. We will survey its major branches through real datasets from medicine, chemistry, and image recognition.

The guiding question is:

> **What information is available during learning, what output is required, and what evidence could support its use?**

## How to use this notebook

1. Read each data card and problem statement before its code.
2. Predict the shape and meaning of each input and output.
3. Read every plot as evidence about a particular representation—not decoration.
4. Treat library calls as implementations of assumptions, not proof that the assumptions are right.
5. Run from top to bottom in the **Rice DSM** kernel.

The datasets are stored in the repository's read-only SQLite teaching database, so the core route works offline. The separate ingestion script records their scikit-learn source. Follow the source links before reusing them: a convenient database is not a substitute for provenance.

## Learning objectives

You will be able to:

- distinguish supervised, unsupervised, and reinforcement learning by feedback signal;
- distinguish regression, classification, clustering, and dimensionality reduction;
- explain where semi-supervised, self-supervised, active, online, transfer, deep, and generative learning fit;
- use `fit`, `predict`, `transform`, and `fit_predict` without confusing their meanings;
- identify the observational unit, representation, target/objective, evaluation evidence, and deployment action;
- distinguish prediction, description, generation, control, and causal inference; and
- record provenance, license, collection process, limitations, and ethical risks before modeling.

## Why this matters in industry

“Use a neural network” does not define a problem. Teams first need to state whether a system predicts a number, assigns a category, discovers structure, compresses a signal, generates an object, or chooses actions over time.

| Principal feedback during learning | Major branch | Typical output |
| --- | --- | --- |
| Features **and observed targets** | Supervised learning | number, category, probability |
| Features, **no supplied target** | Unsupervised learning | representation, cluster, density, anomaly score |
| State, action, and delayed reward | Reinforcement learning | policy or value |

These branches describe the principal learning signal. They are not exclusive product labels: a generative model may use self-supervised pretraining and reinforcement feedback; a semi-supervised method combines labeled and unlabeled observations.

## Historical lens: learning did not begin with deep networks

Several mathematical traditions converged into modern machine learning:

| Period | Contribution | Enduring question |
| --- | --- | --- |
| 1805–1809 | Legendre and Gauss developed least squares for inconsistent astronomical observations | How should noisy measurements determine unknown parameters? |
| 1930s–1950s | Statistical decision theory and pattern recognition connected data, loss, and action | Which rule minimizes expected loss under uncertainty? |
| 1950 | Turing discussed the possibility of constructing a learning machine | Can useful behavior be acquired rather than exhaustively programmed? |
| 1958–1959 | Rosenblatt's perceptron and Samuel's checkers program made adaptive computation concrete | How can experience change future predictions or play? |
| 1980s–present | Larger datasets, automatic differentiation, accelerators, and networked software expanded scale | How do we train, evaluate, deploy, and govern learned systems reliably? |

This is not a march in which every new method replaces the previous one. Least squares, nearest neighbors, trees, probabilistic models, and neural networks solve different problems and remain useful. Modern scale changes what can be represented and optimized; it does not eliminate sampling assumptions, objectives, or scientific judgment.

**Primary historical anchors:** Turing, *Computing Machinery and Intelligence* (1950); Rosenblatt, *The Perceptron* (1958); Samuel, *Some Studies in Machine Learning Using the Game of Checkers* (1959).

In [ ]:
from __future__ import annotations

import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 438
rng = np.random.default_rng(RANDOM_SEED)
assert tuple(int(part) for part in sklearn.__version__.split(".")[:2]) >= (1, 7)
print(f"scikit-learn {sklearn.__version__}; seed={RANDOM_SEED}")

## A problem-first map

Before choosing an estimator, write:

1. **Unit and population:** what is one row, image, sequence, or episode; which future cases matter?
2. **Available information:** what can legitimately be known when the output is needed?
3. **Learning signal:** target, absence of target, reward, or a combination.
4. **Output and action:** number, label, representation, grouping, generated object, or policy—and who consumes it.
5. **Evidence:** held-out targets, stability, reconstruction behavior, intervention return, expert review, or another justified criterion.
6. **Data governance:** origin, consent/authority, license, sensitive fields, collection bias, and allowed uses.

The same data can support multiple mathematical tasks, but not every claim. Prediction from observational data is not automatically causal evidence about an intervention.

## The mathematical anatomy of a learning problem

Let $D$ denote observed experience and let $\mathcal{F}$ be a collection of candidate functions. A learning algorithm is itself a map

$$
\mathcal{A}:D\longmapsto \widehat f_D\in\mathcal{F}.
$$

The fitted function depends on the sample; a new sample can produce a new function. Before calling `fit`, locate six choices:

1. **Representation** $\phi$: raw record $r\mapsto x=\phi(r)$.
2. **Hypothesis class** $\mathcal{F}$: which behaviors can be represented?
3. **Objective**: what counts as a better fit, representation, sample, or action?
4. **Algorithm** $\mathcal{A}$: how is one candidate selected or updated?
5. **Evaluation distribution**: to which future cases should evidence transfer?
6. **Decision path**: what person or system consumes the output?

For supervised learning, empirical risk often has the form

$$
\widehat R_D(f)=\frac1n\sum_{i=1}^n L\bigl(y_i,f(x_i)\bigr).
$$

For KMeans, the objective instead measures squared distance to assigned centers. For PCA, it can be expressed through reconstruction error or retained variance. The word “learning” does not imply one universal loss.

Finite observations cannot determine behavior everywhere. The representation, function class, loss, regularization, and optimizer encode **inductive bias**—the preferences that make one continuation beyond the data more likely than another. There is no assumption-free learner.

## Worked examples: real-data gallery and provenance

We will query four real datasets from `data/course_datasets.sqlite`. The database was built from datasets distributed with the locked scikit-learn version:

| Domain | Dataset | Unit | Features | Example task |
| --- | --- | --- | --- | --- |
| Clinical research | Diabetes | patient | 10 baseline variables | regression |
| Medical imaging | Wisconsin Diagnostic Breast Cancer | sampled breast mass | 30 image-derived measurements | classification |
| Analytical chemistry | Wine recognition | wine sample | 13 chemical measurements | clustering |
| Computer vision | Optical handwritten digits | 8×8 image | 64 pixel intensities | dimensionality reduction |

These are real observations, but they are small historical teaching/benchmark datasets. None is automatically representative of a modern deployment population. The breast-cancer example is an algorithm demonstration, **not medical advice or a diagnostic device**.

The first query reads the catalog rather than guessing table meanings. Notice the read-only URI: analysis code should not silently mutate the shared source database.

In [ ]:
database_path = Path("data/course_datasets.sqlite")
if not database_path.is_file():
    raise FileNotFoundError(
        "Missing data/course_datasets.sqlite. Open the repository root in VS Code "
        "or rebuild it with: uv run python scripts/build_course_database.py"
    )
course_database = sqlite3.connect(f"file:{database_path}?mode=ro", uri=True)
dataset_summary = pd.read_sql_query(
    """
    SELECT dataset_id, domain, task, observational_unit,
           row_count AS observations, feature_count AS features, source_url
    FROM dataset_catalog
    ORDER BY dataset_id
    """,
    course_database,
)
assert dataset_summary["observations"].min() > 100
dataset_summary

## 1. Supervised regression: predict a quantitative target

The diabetes data contain ten baseline variables and a quantitative measure of disease progression one year later. Regression learns from pairs $(\boldsymbol x_i,y_i)$ with numeric $y_i$.

We use only body mass index in the first picture so the relationship is visible, then evaluate on observations excluded from fitting. The fitted line estimates an association useful for prediction under assumptions; it does not prove that changing BMI would cause the predicted change.

In [ ]:
diabetes = pd.read_sql_query(
    """
    SELECT observation_id, age, sex, bmi, bp, s1, s2, s3, s4, s5, s6,
           disease_progression
    FROM diabetes_observations
    ORDER BY observation_id
    """,
    course_database,
)
diabetes_train, diabetes_test = train_test_split(
    diabetes, test_size=0.25, random_state=RANDOM_SEED
)
diabetes_model = LinearRegression().fit(
    diabetes_train[["bmi"]], diabetes_train["disease_progression"]
)
diabetes_prediction = diabetes_model.predict(diabetes_test[["bmi"]])
diabetes_mae = mean_absolute_error(
    diabetes_test["disease_progression"], diabetes_prediction
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].scatter(diabetes_train["bmi"], diabetes_train["disease_progression"], alpha=0.55, label="training patients")
bmi_domain = np.linspace(diabetes_train["bmi"].min(), diabetes_train["bmi"].max(), 120)
axes[0].plot(bmi_domain, diabetes_model.predict(pd.DataFrame({"bmi": bmi_domain})), color="crimson", label="fitted line")
axes[0].set(xlabel="baseline BMI (kg/m²)", ylabel="one-year progression score", title="Regression learns a numeric response")
axes[0].legend()
axes[1].scatter(diabetes_test["disease_progression"], diabetes_prediction, alpha=0.65)
limits = [diabetes_test["disease_progression"].min(), diabetes_test["disease_progression"].max()]
axes[1].plot(limits, limits, color="black", linestyle="--")
axes[1].set(xlabel="observed score", ylabel="predicted score", title=f"Held-out MAE = {diabetes_mae:.1f} points")
plt.tight_layout()
plt.show()

assert len(diabetes_prediction) == len(diabetes_test)
assert diabetes_mae < 70

## 2. Supervised classification: predict a category

The Wisconsin Diagnostic Breast Cancer data contain features computed from digitized images of fine-needle aspirates. Classification learns a discrete target. Many classifiers first produce scores or estimated probabilities, then apply a decision threshold.

We fit all 30 features but separately plot two measurements to show overlap. Accuracy and a confusion matrix demonstrate interfaces only; clinical use would demand carefully defined sensitivity/specificity costs, calibration, external validation, subgroup audits, workflow study, and regulatory evidence.

In [ ]:
cancer = pd.read_sql_query(
    "SELECT * FROM breast_cancer_observations ORDER BY observation_id",
    course_database,
)
cancer_train, cancer_test = train_test_split(
    cancer, test_size=0.25, random_state=RANDOM_SEED, stratify=cancer["diagnosis_code"]
)
cancer_features = [
    column for column in cancer.columns
    if column not in {"observation_id", "diagnosis_code", "diagnosis_label"}
]
cancer_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2_000))
cancer_model.fit(cancer_train[cancer_features], cancer_train["diagnosis_code"])
cancer_prediction = cancer_model.predict(cancer_test[cancer_features])
cancer_accuracy = accuracy_score(cancer_test["diagnosis_code"], cancer_prediction)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
cancer_target_names = np.array(["malignant", "benign"])
for class_value, class_name in enumerate(cancer_target_names):
    subset = cancer_train[cancer_train["diagnosis_code"] == class_value]
    axes[0].scatter(subset["mean_radius"], subset["mean_texture"], alpha=0.5, label=class_name)
axes[0].set(xlabel="mean radius", ylabel="mean texture", title="Two of 30 image-derived features")
axes[0].legend()
ConfusionMatrixDisplay.from_predictions(
    cancer_test["diagnosis_code"], cancer_prediction,
    display_labels=cancer_target_names, colorbar=False, ax=axes[1],
)
axes[1].set_title(f"Held-out confusion matrix; accuracy={cancer_accuracy:.3f}")
plt.tight_layout()
plt.show()

assert set(cancer_prediction) <= {0, 1}
assert cancer_accuracy > 0.90

## 3. Unsupervised clustering: search for groups without labels

The wine data contain 13 chemical measurements from 178 samples and known cultivar labels. `KMeans` will **not** receive those labels. It partitions a scaled feature space by within-cluster squared Euclidean distance.

We use the known cultivar only afterward as an audit aid. Agreement is neither guaranteed nor the definition of successful clustering: scientific structure may differ from the recorded label, and preprocessing plus distance metric partly define what “similar” means.

In [ ]:
wine = pd.read_sql_query(
    "SELECT * FROM wine_observations ORDER BY observation_id", course_database
)
wine_feature_names = [
    column for column in wine.columns
    if column not in {"observation_id", "cultivar_code", "cultivar_label"}
]
wine_features = wine[wine_feature_names]
wine_scaled = StandardScaler().fit_transform(wine_features)
wine_coordinates = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(wine_scaled)
wine_clusters = KMeans(n_clusters=3, random_state=RANDOM_SEED, n_init=20).fit_predict(wine_scaled)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), sharex=True, sharey=True)
first = axes[0].scatter(wine_coordinates[:, 0], wine_coordinates[:, 1], c=wine_clusters, cmap="viridis", s=38)
axes[0].set(title="KMeans groups (labels hidden during fit)", xlabel="principal component 1", ylabel="principal component 2")
second = axes[1].scatter(wine_coordinates[:, 0], wine_coordinates[:, 1], c=wine["cultivar_code"], cmap="viridis", s=38)
axes[1].set(title="Known cultivar labels (audit only)", xlabel="principal component 1")
fig.colorbar(first, ax=axes[0], label="cluster ID")
fig.colorbar(second, ax=axes[1], label="cultivar ID")
plt.tight_layout()
plt.show()

assert wine_clusters.shape == (178,)
assert np.unique(wine_clusters).size == 3

## 4. Dimensionality reduction: learn a representation

Each handwritten digit is an $8\times8$ image represented by 64 pixel intensities. Principal component analysis (PCA) finds orthogonal directions of greatest variance; it receives no digit labels.

A two-dimensional projection is useful for seeing broad structure but cannot preserve every distance or class boundary from 64 dimensions. Apparent separation in a plot is exploratory evidence, not held-out classification performance.

In [ ]:
digits = pd.read_sql_query(
    "SELECT * FROM digits_observations ORDER BY observation_id", course_database
)
digit_feature_names = [f"pixel_{index}" for index in range(64)]
digit_features = digits[digit_feature_names].to_numpy()
digit_images = digit_features.reshape(-1, 8, 8)

fig, axes = plt.subplots(2, 6, figsize=(10, 3.6))
for ax, image, label in zip(axes.flat, digit_images[:12], digits["digit_label"][:12], strict=True):
    ax.imshow(image, cmap="gray_r")
    ax.set_title(f"label {label}")
    ax.axis("off")
fig.suptitle("Real 8×8 handwritten-digit images")
plt.tight_layout()
plt.show()

digits_scaled = StandardScaler().fit_transform(digit_features)
digits_pca = PCA(n_components=2, random_state=RANDOM_SEED)
digit_coordinates = digits_pca.fit_transform(digits_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    digit_coordinates[:, 0], digit_coordinates[:, 1],
    c=digits["digit_label"], cmap="tab10", s=12, alpha=0.7,
)
ax.set(xlabel="principal component 1", ylabel="principal component 2", title="PCA representation; color labels are for interpretation")
fig.colorbar(scatter, ax=ax, ticks=range(10), label="digit label")
plt.show()

assert digit_coordinates.shape == (1797, 2)
assert 0 < digits_pca.explained_variance_ratio_.sum() < 1

## 5. Reinforcement learning: actions change later data

In reinforcement learning, an agent observes a state, selects an action, and receives a possibly delayed reward. The output is a policy. Unlike an ordinary fixed dataset, the policy changes which observations are collected.

A clean real-data demonstration requires a logged policy, action propensities, rewards, and assumptions for off-policy evaluation—or a safe interactive environment. We therefore use a **declared simulation** for mechanism, not a synthetic table disguised as empirical evidence: three instruments have unknown success probabilities and an epsilon-greedy agent must balance exploration with exploitation.

In [ ]:
true_success_probability = np.array([0.28, 0.47, 0.39])
action_count = np.zeros(3, dtype=int)
reward_sum = np.zeros(3, dtype=float)
cumulative_reward = []
actions = []

for step in range(500):
    estimated_value = np.divide(reward_sum, action_count, out=np.zeros(3), where=action_count > 0)
    explore = rng.random() < 0.10 or np.any(action_count == 0)
    action = int(rng.integers(3) if explore else np.argmax(estimated_value))
    reward = float(rng.random() < true_success_probability[action])
    action_count[action] += 1
    reward_sum[action] += reward
    actions.append(action)
    cumulative_reward.append(reward + (cumulative_reward[-1] if cumulative_reward else 0))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(range(3), action_count)
axes[0].set(xlabel="instrument", ylabel="times selected", title="The policy changes data collection")
axes[1].plot(cumulative_reward)
axes[1].set(xlabel="decision step", ylabel="cumulative reward", title="Reward accumulated through interaction")
plt.tight_layout()
plt.show()

assert sum(action_count) == 500
assert action_count[np.argmax(true_success_probability)] == action_count.max()
course_database.close()

## Related paradigms are different axes

- **Semi-supervised learning:** a small labeled set plus a larger unlabeled set.
- **Self-supervised learning:** targets are constructed from the data, such as masked tokens or image regions.
- **Generative modeling:** learn a distribution or conditional distribution to create samples, text, images, molecules, or other objects.
- **Active learning:** choose which expensive label to request next.
- **Online learning:** update as observations arrive rather than in one fixed batch.
- **Transfer learning:** adapt representations or parameters learned on one task/domain to another.
- **Deep learning:** use multilayer parameterized architectures; it can appear in supervised, self-supervised, generative, or reinforcement settings.

Also distinguish **causal inference**. Predictive association asks what output is likely given observed features. A causal claim asks what would change under an intervention and needs additional design or assumptions.

## Application gallery: the branch does not define the whole project

| Domain | Possible task | Output | Evidence that would matter |
| --- | --- | --- | --- |
| Astronomy | Classify transient events from image and light-curve features | class probabilities | later spectroscopic labels, time/site shift, calibration |
| Materials | Predict later battery capacity from early cycles | capacity with uncertainty | unseen cells, later batches, new chemistries |
| Ecology | Estimate species occurrence from surveys and remote sensing | occurrence probability | spatial holdout, detection process, field validation |
| Manufacturing | Detect unusual sensor trajectories | anomaly score | maintenance outcomes, equipment generations, false-alarm burden |
| Chemistry | Propose molecules satisfying property constraints | ranked/generated candidates | validity, novelty, synthesizability, laboratory confirmation |
| Operations | Choose inventory under uncertain demand | decision policy | realized cost under temporal evaluation and constraints |

An algorithm demonstration can show an interface. It cannot by itself establish representativeness, usefulness, safety, or deployment readiness. Each row needs a different observational unit, split, metric, and feedback path.

## Common failure modes

- **Benchmark blindness:** a convenient historical dataset is treated as representative of current deployment.
- **Label mythology:** a target is assumed to be objective truth without examining how and why it was constructed.
- **Unit leakage:** one patient, device, site, author, or time period appears on both sides of evaluation.
- **Unsupervised overclaiming:** visually pleasing clusters are called natural kinds without stability or domain evidence.
- **Metric substitution:** an easy score replaces the actual decision cost.
- **License/provenance loss:** a CSV is copied without origin, version, authority, or permitted use.
- **Causal language:** a predictive coefficient or feature importance is presented as an intervention effect.

## Debugging and data triage

1. Print shapes, dtypes, target values, missingness, and duplicated units.
2. Read the data card, original source, and license before interpreting column names.
3. Plot raw distributions and representative observations—not only a model score.
4. Split the unit that must generalize before target-guided analysis.
5. Fit transformations inside the training boundary.
6. Compare with a trivial baseline and inspect errors by meaningful slice.
7. Ask whether the dataset contains the metadata needed to evaluate the intended claim.

## Professional practice

Every project should keep a small data-source record:

- stable landing page and dataset identifier;
- creator, steward, citation, and date/version accessed;
- license or usage terms;
- collection and sampling process;
- meaning of one observation and every target;
- units, valid ranges, missing-value codes, and transformations;
- sensitive attributes and foreseeable harms;
- known limitations and prohibited claims; and
- checksum or immutable snapshot reference when redistribution is permitted.

A model registry without a data lineage record is incomplete.

## Guided practice: formulate before fitting

Choose the digits or wine dataset and write two different ML problems that use the same rows. For each, state unit, features available at prediction time, learning signal, output, action, and evaluation evidence.

**Success criteria:** the two tasks must make different claims; one must be unsupervised; and neither may use a label while claiming it was unavailable.

In [ ]:
# A mechanical provenance checkpoint to extend in your own work.
provenance_record = {
    "name": "Optical recognition of handwritten digits",
    "table": "digits_observations",
    "source": "UCI ML handwritten digits test set",
    "observational_unit": "one 8x8 processed handwritten-digit image",
    "features": 64,
    "target": "digit identity 0-9 (not used by PCA fitting)",
}
assert provenance_record["features"] == digit_features.shape[1]
provenance_record

## Independent practice

Select a new real dataset from one of the repositories below. Create a one-page data card and three visualizations before fitting a model: distribution of the proposed target, distribution/support of key features, and one plot that could reveal selection, missingness, time, group, or class imbalance.

**Success criteria:** link the primary landing page; record license and version/date; define the row and target; identify at least one unsupported claim; and keep the raw download out of Git if redistribution terms or size make that inappropriate.

## Extension: compare domains without comparing scores

Choose two datasets from different domains. Explain why their metric values cannot be compared as if they measured task difficulty. Examine sample construction, target ambiguity, feature acquisition cost, class/target distribution, and consequences of error.

## Where to find public data

Start with sources that preserve metadata and a stable landing page:

- [UCI Machine Learning Repository](https://archive.ics.uci.edu/) — curated ML datasets across many domains.
- [OpenML](https://www.openml.org/) — versioned datasets, tasks, and benchmark runs.
- [Data.gov](https://data.gov/) — United States government open-data catalog.
- [NASA Science Data](https://science.data.nasa.gov/) and [NOAA Data Discovery](https://data.noaa.gov/onestop/) — space, Earth, weather, ocean, and climate data.
- [NIH data repositories](https://sharing.nih.gov/data-management-and-sharing-policy/sharing-scientific-data/repositories-for-sharing-scientific-data) — repository guidance for biomedical data.
- [World Bank Open Data](https://data.worldbank.org/) — global development indicators.
- [Google Dataset Search](https://datasetsearch.research.google.com/) — a search engine; verify the underlying steward and license yourself.

“Publicly reachable” does not necessarily mean openly licensed, ethically appropriate, representative, clean, or safe to redistribute.

## Retrieval practice

1. What distinguishes the three major branches by feedback signal?
2. How do regression and classification differ?
3. Why can known wine labels be used to audit but not fit the clustering example?
4. What information is lost in a two-component PCA plot?
5. Why is a simulation appropriate for the bandit mechanism but not empirical evidence about real instruments?
6. Why is prediction not automatically causal inference?
7. Which provenance fields must travel with a dataset?

## Takeaway

Start with the observational unit, learning signal, output, action, and evidence—not an algorithm name. Real data make domain assumptions and limitations visible, but “real” does not imply representative, ethical, or deployment-ready. The mathematical branch and the data lineage jointly determine what can be claimed.

**Next:** the supervised-learning notebook derives ordinary least squares and then builds a complete held-out regression pipeline using the real diabetes data.

## Further reading

- [scikit-learn: toy dataset descriptions and original sources](https://scikit-learn.org/stable/datasets/toy_dataset.html)
- [scikit-learn: getting started](https://scikit-learn.org/stable/getting_started.html)
- [scikit-learn: common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html)
- [NumPy array fundamentals](https://numpy.org/doc/stable/user/absolute_beginners.html)
- [Python tutorial: data structures](https://docs.python.org/3/tutorial/datastructures.html)